In [1]:
import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer
from pandas_ml import ConfusionMatrix

In [2]:
train = pd.read_csv('UNSW_NB15_training-set.csv')
test = pd.read_csv('UNSW_NB15_testing-set.csv')
combined_data = pd.concat([train, test]).drop(['id'],axis=1)

In [3]:
# Contaminsation mean pollution (outliers) in data
tmp = train.where(train['attack_cat'] == "Normal").dropna()
contamination = round(1 - len(tmp)/len(train), 2)
print("train contamination ", contamination)

tmp = test.where(test['attack_cat'] == "Normal").dropna()
print("test  contamination ", round(1 - len(tmp)/len(test),2),'\n')

if contamination > 0.5:
    print(f'contamination is {contamination}, which is greater than 0.5. Fixing...')
    contamination = round(1-contamination,2)
    print(f'contamination is now {contamination}')

train contamination  0.55
test  contamination  0.68 

contamination is 0.55, which is greater than 0.5. Fixing...
contamination is now 0.45


In [4]:
from sklearn.preprocessing import LabelEncoder,normalize
le1 = LabelEncoder()
le = LabelEncoder()

vector = combined_data['attack_cat']

print("attack cat:", set(list(vector))) # use print to make it print on single line 

combined_data['attack_cat'] = le1.fit_transform(vector)
combined_data['proto'] = le.fit_transform(combined_data['proto'])
combined_data['service'] = le.fit_transform(combined_data['service'])
combined_data['state'] = le.fit_transform(combined_data['state'])

vector = combined_data['attack_cat']
print('\nDescribing attack_type: ')
print("min", vector.min())
print("max", vector.max())
print("mode",vector.mode(), "Which is,", le1.inverse_transform(vector.mode()))
print("mode", len(np.where(vector.values==6)[0])/len(vector),"%")

attack cat: {'Generic', 'Analysis', 'Normal', 'Shellcode', 'Exploits', 'Reconnaissance', 'Backdoor', 'Worms', 'Fuzzers', 'DoS'}

Describing attack_type: 
min 0
max 9
mode 0    6
dtype: int32 Which is, ['Normal']
mode 0.3609225646458884 %


In [5]:
le1.inverse_transform([0,1,2,3,4,5,6,7,8,9])
combined_data.head(3)

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,0.000011,119,0,5,2,0,496,0,90909.0902,254,...,1,2,0,0,0,1,2,0,6,0
1,0.000008,119,0,5,2,0,1762,0,125000.0003,254,...,1,2,0,0,0,1,2,0,6,0
2,0.000005,119,0,5,2,0,1068,0,200000.0051,254,...,1,3,0,0,0,1,3,0,6,0


In [6]:
## OMITTED: For statistical feature removal

lowSTD = list(combined_data.std().to_frame().nsmallest(6, columns=0).index)
# this is stupid. suppose a feature has a 1.0 (spearman or pearson) correlation, OR conditional probability, when not 0.... That a very useful feature  

lowCORR = list(combined_data.corr().abs().sort_values('attack_cat')['attack_cat'].nsmallest(3).index) # .where(lambda x: x < 0.005).dropna()
# This might be stupid. A Deep MLP (feed forward neural net) may see patterns

drop = set( lowCORR + lowSTD)
drop = {'ackdat', 'ct_ftp_cmd', 'djit', 'is_ftp_login', 'is_sm_ips_ports', 'response_body_len', 'sjit', 'synack', 'tcprtt'}
# print(f'Before {combined_data.shape}')
combined_data_reduced=combined_data.drop(drop,axis=1)
# print(f'After {combined_data.shape}')

In [17]:
data_x = combined_data_reduced.drop(['attack_cat','label'], axis=1) # droped label
data_y = combined_data_reduced.loc[:,['label']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO

y_train = y_train.values.flatten()
y_test = y_test.values.flatten()

In [18]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape) # test is larger... good 
print(y_test.shape)
# y_train.min()

(206138, 33)
(206138,)
(51535, 33)
(51535,)


In [19]:
# traindata = pd.read_csv('UNSW_NB15_training-set.csv', header=None)
# testdata = pd.read_csv('UNSW_NB15_testing-set.csv', header=None)
# traindata = pd.read_csv('kddtrain.csv', header=None)
# testdata = pd.read_csv('kddtest.csv', header=None)

# X = traindata.iloc[:,1:42]
# Y = traindata.iloc[:,0]
# C = testdata.iloc[:,0]
# T = testdata.iloc[:,1:42]
X = X_train
Y = y_train
C = y_test
T = X_test

scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)


traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)


model = LogisticRegression()
model.fit(traindata, trainlabel)


# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print("***************************************************************")


D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


(51535,)
(51535,)
***************************************************************


In [20]:
model = LogisticRegression()
model.fit(traindata, trainlabel)
print(model)

# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)


cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='warn',
          n_jobs=None, penalty='l2', random_state=None, solver='warn',
          tol=0.0001, verbose=0, warm_start=False)
(51535,)
(51535,)
population: 51535
P: 32860
N: 18675
PositiveTest: 28155
NegativeTest: 23380
TP: 24121
TN: 14641
FP: 4034
FN: 8739
TPR: 0.73405356056
TNR: 0.783989290495
PPV: 0.856721719055
NPV: 0.62621899059
FPR: 0.216010709505
FDR: 0.143278280945
FNR: 0.26594643944
ACC: 0.752149024935
F1_score: 0.790658034909
MCC: 0.500183948278
informedness: 0.518042851055
markedness: 0.482940709645
prevalence: 0.637624915106
LRP: 3.39822762604
LRN: 0.339222031046
DOR: 10.0177090962
FOR: 0.37378100941
Predicted  False   True  __all__
Actual                          
False      14641   4034    18675
True        8739  24121    32860
__all__    23380  28155    51535
(51535,)
(51535,)
***********************************************************

In [ ]:
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.wrappers.scikit_learn import KerasClassifier
import h5py
from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from keras.utils import to_categorical

def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=33,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

#DNN
batch_size = 64
checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
#model = KerasClassifier(build_fn=build_model, epochs=2, batch_size=batch_size)
model = KerasClassifier(build_fn=build_model, epochs=10, batch_size=batch_size)
model.fit(traindata, trainlabel,batch_size=batch_size, epochs=40, callbacks=[checkpointer,csv_logger])
#model.save("DNNResult/dnn1layer_model.hdf5")

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
predicted = predicted.flatten()
cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")


Epoch 1/40
206138/206138 [==============================] - 12s 57us/step - loss: 0.4438 - acc: 0.7516

Epoch 00001: loss improved from inf to 0.44375, saving model to ./DNNResult/checkpoint-01.hdf5
Epoch 2/40
206138/206138 [==============================] - 12s 57us/step - loss: 0.4242 - acc: 0.7753

Epoch 00002: loss improved from 0.44375 to 0.42423, saving model to ./DNNResult/checkpoint-02.hdf5
Epoch 3/40
206138/206138 [==============================] - 11s 55us/step - loss: 0.4079 - acc: 0.7940

Epoch 00003: loss improved from 0.42423 to 0.40788, saving model to ./DNNResult/checkpoint-03.hdf5
Epoch 4/40
206138/206138 [==============================] - 11s 55us/step - loss: 0.3955 - acc: 0.8059

Epoch 00004: loss improved from 0.40788 to 0.39546, saving model to ./DNNResult/checkpoint-04.hdf5
Epoch 5/40
206138/206138 [==============================] - 12s 56us/step - loss: 0.3876 - acc: 0.8099

Epoch 00005: loss improved from 0.39546 to 0.38763, saving model to ./DNNResult/checkpoi

In [10]:



# fit a Naive Bayes model to the data
model = GaussianNB()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)

expected = expected.flatten()
#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



# fit a k-nearest neighbor model to the data
model = KNeighborsClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")
model = DecisionTreeClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()


print("***************************************************************")


print("AdaBoostClassifier(n_estimators=100)")

model = AdaBoostClassifier(n_estimators=100)
model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



print("RandomForestClassifier(n_estimators=100)")
model = RandomForestClassifier(n_estimators=100)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")

print("svm.SVC(kernel='linear')")
model = svm.SVC(kernel='linear')#调参
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")

D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GaussianNB(priors=None, var_smoothing=1e-09)
(51535,)
(51535,)
population: 51535
P: 32860
N: 18675
PositiveTest: 47500
NegativeTest: 4035
TP: 32769
TN: 3944
FP: 14731
FN: 91
TPR: 0.997230675593
TNR: 0.211191432396
PPV: 0.689873684211
NPV: 0.977447335812
FPR: 0.788808567604
FDR: 0.310126315789
FNR: 0.00276932440657
ACC: 0.71238963811
F1_score: 0.815555002489
MCC: 0.372940281679
informedness: 0.20842210799
markedness: 0.667321020022
prevalence: 0.637624915106
LRP: 1.26422394045
LRN: 0.0131128634109
DOR: 96.410974539
FOR: 0.0225526641884
Predicted  False   True  __all__
Actual                          
False       3944  14731    18675
True          91  32769    32860
__all__     4035  47500    51535
(51535,)
(51535,)
***************************************************************


D:\Anaconda3\envs\TF_36a\lib\site-packages\ipykernel_launcher.py:36: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().


KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=5, p=2,
           weights='uniform')
(51535,)
(51535,)
population: 51535
P: 32860
N: 18675
PositiveTest: 34510
NegativeTest: 17025
TP: 29325
TN: 13490
FP: 5185
FN: 3535
TPR: 0.892422398052
TNR: 0.722356091031
PPV: 0.849753694581
NPV: 0.792364170338
FPR: 0.277643908969
FDR: 0.150246305419
FNR: 0.107577601948
ACC: 0.830794605608
F1_score: 0.87056553362
MCC: 0.628299491332
informedness: 0.614778489083
markedness: 0.642117864919
prevalence: 0.637624915106
LRP: 3.21426967862
LRN: 0.148925998249
DOR: 21.5829990493
FOR: 0.207635829662
Predicted  False   True  __all__
Actual                          
False      13490   5185    18675
True        3535  29325    32860
__all__    17025  34510    51535
(51535,)
(51535,)
***************************************************************
DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
            m

D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


population: 51535
P: 32860
N: 18675
PositiveTest: 39241
NegativeTest: 12294
TP: 31892
TN: 11326
FP: 7349
FN: 968
TPR: 0.970541692027
TNR: 0.606479250335
PPV: 0.812721388344
NPV: 0.921262404425
FPR: 0.393520749665
FDR: 0.187278611656
FNR: 0.0294583079732
ACC: 0.838614533812
F1_score: 0.884647924439
MCC: 0.650787230807
informedness: 0.577020942361
markedness: 0.733983792769
prevalence: 0.637624915106
LRP: 2.46630372821
LRN: 0.0485726559597
DOR: 50.775558377
FOR: 0.0787375955751
Predicted  False   True  __all__
Actual                          
False      11326   7349    18675
True         968  31892    32860
__all__    12294  39241    51535
(51535,)
(51535,)
***************************************************************
RandomForestClassifier(n_estimators=100)


D:\Anaconda3\envs\TF_36a\lib\site-packages\ipykernel_launcher.py:113: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().


population: 51535
P: 32860
N: 18675
PositiveTest: 38883
NegativeTest: 12652
TP: 32564
TN: 12356
FP: 6319
FN: 296
TPR: 0.990992087645
TNR: 0.661633199465
PPV: 0.837486819433
NPV: 0.976604489409
FPR: 0.338366800535
FDR: 0.162513180567
FNR: 0.00900791235545
ACC: 0.87164063258
F1_score: 0.907795882525
MCC: 0.72890093577
informedness: 0.652625287109
markedness: 0.814091308841
prevalence: 0.637624915106
LRP: 2.92875094742
LRN: 0.0136146619649
DOR: 215.117419366
FOR: 0.0233955105912
Predicted  False   True  __all__
Actual                          
False      12356   6319    18675
True         296  32564    32860
__all__    12652  38883    51535
(51535,)
(51535,)
***************************************************************
svm.SVC(kernel='linear')


D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


population: 51535
P: 32860
N: 18675
PositiveTest: 27719
NegativeTest: 23816
TP: 23867
TN: 14823
FP: 3852
FN: 8993
TPR: 0.726323797931
TNR: 0.793734939759
PPV: 0.861033947834
NPV: 0.622396708095
FPR: 0.206265060241
FDR: 0.138966052166
FNR: 0.273676202069
ACC: 0.750751916173
F1_score: 0.787962825402
MCC: 0.501410347603
informedness: 0.52005873769
markedness: 0.483430655929
prevalence: 0.637624915106
LRP: 3.52131280539
LRN: 0.344795457981
DOR: 10.2127586773
FOR: 0.377603291905
Predicted  False   True  __all__
Actual                          
False      14823   3852    18675
True        8993  23867    32860
__all__    23816  27719    51535
(51535,)
(51535,)
***************************************************************


Using TensorFlow backend.
